### Libraries

In [ ]:
#Cargas
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt


from sklearn.impute import SimpleImputer


def load_data():
    candidates = [
        Path('/content/datos_genero_filtrados.xlsx'),   
        Path('data/raw/datos_genero_filtrados.xlsx'),   
        Path('datos_genero_filtrados.xlsx'),            
    ]
    for p in candidates:
        if p.exists():
            df = pd.read_excel(p)
            print("=== Dataset cargado ===")
            print(f"Archivo: {p} | Filas={df.shape[0]} | Columnas={df.shape[1]}")
            display(df.head(3))
            return df
    raise FileNotFoundError('No se encontró datos_genero_filtrados.xlsx en rutas conocidas.')


df = load_data() 


=== Dataset cargado ===
Archivo: /content/datos_genero_filtrados.xlsx | Filas=235 | Columnas=36


,profesionales,edad,motivo_de_consulta,medio_por_el_que_ingresa,genero,nacionalidad,barrio,municipio,localidad,estado_civil,...,simbolica,ambiental,politica,digital,cant_tipos_violencias_por_persona,denuncio,medidas_de_proteccion,fecha_fin_de_vigencia,personas_a_cargo,red_vincular
0,Fanny / Dana,28.0,VIOLENCIA,NaN,Mujer,Argentina,bordeu,Bahía Blanca,Bahía Blanca,Soltera,...,0.0,0.0,0.0,0.0,2.0,NO,NO,NaN,NO,PARIENTES CONVIVIENTES
1,Agus/ Analé,43.0,VIOLENCIA,NaN,Mujer,Argentina,pacifico,Bahía Blanca,Bahía Blanca,NaN,...,0.0,1.0,0.0,0.0,4.0,SI,Prohibición de acercamiento,NaN,Hija/Hijo,PARIENTES NO CONVIVIENTES
2,Fanny/ Majo,53.0,VIOLENCIA,NaN,Mujer,Argentina,pampa central,Bahía Blanca,Bahía Blanca,Soltera,...,0.0,0.0,0.0,0.0,2.0,NO,NO,NaN,NO,PARIENTES NO CONVIVIENTES


# Universidad Nacional de la Matanza

Specialization in Data Science

Benko Teo
Cura Diego
Riganti Valentina
Sanjuan Oriana.

Model based on data from the Dirección General de Género de Bahía Blanca

# **Stage: Data Cleaning and Preprocessing**

General corrections are performed on categorical variables:
* Text Normalization
* Handling Missing Values
* Standardization

Regarding numerical variables, relevant clarifications are made for each case.

### Analysis of inquiry variables (assigned professional, reason for the inquiry, and referral channel):

In [ ]:
# Columns to be cleaned
col_limpiar = ['profesionales', 'motivo_de_consulta']

for col in col_limpiar:

    # 1. Normalization
    df[col] = df[col].astype(str).str.lower().str.strip()

    # 2. Missing values
    df[col] = df[col].replace({
        'nan': 'desconocido',
        'nulo (sin dato)': 'desconocido',
        'sin datos': 'desconocido',
        'otra': 'desconocido'
    })

    # 3. Standarization
    df[col] = df[col].str.replace('/', '_', regex=False).str.replace(' ', '_', regex=False)


# Verification
print("--- Count of the 'professionals' Variable (Post-Cleaning - Top 5) ---")
print(df['profesionales'].value_counts(dropna=False).head(5).to_markdown())

print("\n--- Count of the variable 'motivo_de_consulta' (Post-Cleaning) ---")
print(df['motivo_de_consulta'].value_counts(dropna=False).to_markdown())

--- Conteo de la Variable 'profesionales' (Post-Limpieza - Top 5) ---
| profesionales   |   count |
|:----------------|--------:|
| ana_gladis      |      12 |
| analé_ana       |       5 |
| agus___rami     |       4 |
| desconocido     |       4 |
| fernanda        |       4 |

--- Conteo de la Variable 'motivo_de_consulta' (Post-Limpieza) ---
| motivo_de_consulta   |   count |
|:---------------------|--------:|
| violencia            |     202 |
| desconocido          |      22 |
| asesoramiento        |       9 |
| socioeconómica       |       2 |


In [ ]:
columna = 'medio_por_el_que_ingresa'

# 1. Normalization
df[columna] = df[columna].astype(str).str.lower().str.strip()

# 2. Null unificarion
df[columna] = df[columna].replace({
    'nan': 'desconocido',
    'sin datos': 'desconocido',
    'nulo (sin dato)': 'desconocido'
})

# 3. Standarization
df[columna] = df[columna].str.replace(' ', '_', regex=False).str.replace('/', '_', regex=False)

# Verification
print("--- Variable Count 'medio_por_el_que_ingresa' (Post-Cleaning) ---")
print(df[columna].value_counts(dropna=False).to_markdown())

--- Conteo de la Variable 'medio_por_el_que_ingresa' (Post-Limpieza) ---
| medio_por_el_que_ingresa                    |   count |
|:--------------------------------------------|--------:|
| notificación_judicial                       |      83 |
| espontánea                                  |      50 |
| desconocido                                 |      48 |
| otra                                        |      26 |
| turno_programado                            |      15 |
| comisaría                                   |       6 |
| unidad_sanitaria                            |       3 |
| organización_social_institución_comunitaria |       2 |
| institución_educativa                       |       1 |
| hospital                                    |       1 |


### Analysis of variables describing the location of the person seeking assistance:

In [ ]:
location_cols = ['nacionalidad', 'barrio', 'municipio', 'localidad']

for col in location_cols:

    # 1. Normalization
    df[col] = df[col].astype(str).str.lower().str.strip()

    # 2. Unification of Nulls
    df[col] = df[col].replace({
        'nan': 'desconocida',          # Nulos de Python
        'nulo (sin dato)': 'desconocida',
        's/d': 'desconocida',
        'sin datos': 'desconocida',
        'sin dato': 'desconocida',
    })

    # 3. Specific Correction of Geographic Inconsistencies (Bahía Blanca)
    if col in ['municipio', 'localidad']:
        df[col] = df[col].replace({
            'bahia blanca': 'bahia_blanca', 
            'bahía blanca': 'bahia_blanca'
        })

    # 4. Standarization
    df[col] = df[col].str.replace(' ', '_', regex=False).str.replace('/', '_', regex=False)


# Verification
print("--- Count of 'nacionalidad' ---")
print(df['nacionalidad'].value_counts(dropna=False).to_markdown())

print("\n--- Count of  'municipio' ---")
print(df['municipio'].value_counts(dropna=False).to_markdown())

print("\n--- Count of 'localidad' ---")
print(df['localidad'].value_counts(dropna=False).to_markdown())

print("\n--- Count of 'barrio' ---")
print(df['barrio'].value_counts(dropna=False).head(5).to_markdown())

--- Conteo de la Variable 'nacionalidad' (Post-Limpieza) ---
| nacionalidad   |   count |
|:---------------|--------:|
| argentina      |     219 |
| desconocida    |      13 |
| chilena        |       2 |
| no             |       1 |

--- Conteo de la Variable 'municipio' (Inconsistencias Corregidas - Post-Limpieza) ---
| municipio    |   count |
|:-------------|--------:|
| bahia_blanca |     231 |
| desconocida  |       2 |
| punta_alta   |       1 |
| buenos_aires |       1 |

--- Conteo de la Variable 'localidad' (Inconsistencias Corregidas - Post-Limpieza) ---
| localidad          |   count |
|:-------------------|--------:|
| bahia_blanca       |     228 |
| gral._daniel_cerri |       3 |
| desconocida        |       1 |
| punta_alta         |       1 |
| buenos_aires       |       1 |
| ing._white         |       1 |

--- Conteo de la Variable 'barrio' (Top 5 - Post-Limpieza) ---
| barrio          |   count |
|:----------------|--------:|
| desconocida     |      20 |
| noroest

### Analysis of variables describing the person's characteristics:

In [ ]:
col_genero = 'genero'

# 1. Normalization
df[col_genero] = df[col_genero].astype(str).str.lower().str.strip()

# 2. Unification of Nulls
df[col_genero] = df[col_genero].replace({
    'nan': 'desconocido',          
    'nulo (sin dato)': 'desconocido',
    'sin datos': 'desconocido'
})

# --- Verification ---
print("--- Count of 'genero' ---")
print(df[col_genero].value_counts(dropna=False).to_markdown())

--- Conteo de la Variable Limpia 'genero' ---
| genero      |   count |
|:------------|--------:|
| mujer       |     228 |
| desconocido |       6 |
| otro        |       1 |


For the numerical variable 'age' ('edad'), missing values are imputed using the median.

In [ ]:
# Median Imputation for Missing Values
median_edad = df['edad'].median()
df['edad'] = df['edad'].fillna(median_edad)

# Conversion of Variable to Integer (int)
df['edad'] = df['edad'].astype(int)

print("\n---    Verification of the column 'edad' (Post-Cleaning) ---")
print(f"Total Missing Values in 'edad': {df['edad'].isnull().sum()} nulos")

print("\n--- Descriptive Statistics 'edad' (Post-Cleaning) ---")
print(df['edad'].describe().to_markdown())

print("\n--- Variable Type 'edad' (Post-Cleaning) ---")
df['edad'].info()


--- Verificación de la Columna 'edad' (Post-Limpieza) ---
Total de Nulos en 'edad': 0 nulos

--- Resumen Estadístico 'edad' (Post-Limpieza) ---
|       |     edad |
|:------|---------:|
| count | 235      |
| mean  |  38.1532 |
| std   |  12.3334 |
| min   |  17      |
| 25%   |  29.5    |
| 50%   |  36      |
| 75%   |  45      |
| max   |  85      |

--- Tipo de Variable 'edad' (Post-Limpieza) ---
<class 'pandas.core.series.Series'>
RangeIndex: 235 entries, 0 to 234
Series name: edad
Non-Null Count  Dtype
--------------  -----
235 non-null    int64
dtypes: int64(1)
memory usage: 2.0 KB


### Analysis of socioeconomic and status variables:

In [ ]:
# Socioeconomic and Status Columns to be cleaned
socio_cols = [
    'estado_civil', 'nivel_educativo', 'situacion_laboral',
    'percibe_prestacion_estatal', 'vivienda', 'obra_social'
]

for col in socio_cols:

    # 1. Normalization
    df[col] = df[col].astype(str).str.lower().str.strip()

    # 2. Unification of Nulls
    df[col] = df[col].replace({
        'nan': 'desconocido',          # Nulos de Python
        'nulo (sin dato)': 'desconocido',
        'sin datos': 'desconocido',
        'sin dato': 'desconocido',
        's/d': 'desconocido'
    })

# Specific Text Inconsistency Corrections

# 3. 'nivel_educativo': Typo Correction and Inconsistency Grouping
df['nivel_educativo'] = df['nivel_educativo'].replace({
    'primario incompletp': 'primario incompleto',  
    'univrtsitario incompleto': 'universitario incompleto', 
})

# 4. 'situacion_laboral': Case Correction and Similarity Unification
df['situacion_laboral'] = df['situacion_laboral'].replace({
    'trabaja formal': 'trabajo formal',
})
# Note: Imputation of null/unknown values was already performed in the general loop.

# 5. 'percibe_prestacion_estatal'
df['percibe_prestacion_estatal'] = df['percibe_prestacion_estatal'].replace({
'cuota alimentaria': 'no percibe' #not a benefit
})

# Verification
print("--- Count of 'estado_civil' (Post-Cleaning)---")
print(df['estado_civil'].value_counts(dropna=False).to_markdown())

print("\n--- Count of 'nivel_educativo' (Post-Cleaning) ---")
print(df['nivel_educativo'].value_counts(dropna=False).to_markdown())

print("\n---Count of 'situacion_laboral' (Post-Cleaning) ---")
print(df['situacion_laboral'].value_counts(dropna=False).to_markdown())

print("\n--- Count of 'percibe_prestacion_estatal' (Post-Cleaning) ---")
print(df['percibe_prestacion_estatal'].value_counts(dropna=False).to_markdown())

print("\n--- Count of 'vivienda' (Post-Cleaning) ---")
print(df['vivienda'].value_counts(dropna=False).to_markdown())

print("\n--- CCount of 'obra_social' (Post-Cleaning) ---")
print(df['obra_social'].value_counts(dropna=False).to_markdown())

--- Conteo de la Variable 'estado_civil' (Post-Limpieza)---
| estado_civil       |   count |
|:-------------------|--------:|
| soltera            |      97 |
| casada             |      50 |
| separada           |      33 |
| desconocido        |      25 |
| unión convivencial |      13 |
| divorciada         |      10 |
| viuda              |       7 |

--- Conteo de la Variable 'nivel_educativo' (Post-Limpieza) ---
| nivel_educativo                    |   count |
|:-----------------------------------|--------:|
| secundario incompleto              |      67 |
| secundario completo                |      40 |
| desconocido                        |      30 |
| terciario completo                 |      21 |
| primario completo                  |      19 |
| terciario incompleto               |      11 |
| primario incompleto                |      10 |
| universitario completo             |      10 |
| terciario en curso                 |       7 |
| universitario incompleto           | 

### Analysis of Clinical, Risk, and Family descriptors:

In [ ]:

cols_clinicas_familiares = [
    'diagnostico', 'tratamiento', 'posee_cud',
    'hijos_pea', 'convivencia_pea'
]

for col in cols_clinicas_familiares:

    # 1. Normalization
    df[col] = df[col].astype(str).str.lower().str.strip()

    # 2. Standarization
    # the 'Option 1' issue is resolved by addressing the hard character separation
    df[col] = df[col].str.replace(r'[^\S\n\r]', ' ', regex=True).str.strip()
    # then proceed normally
    if col in ['diagnostico', 'tratamiento', 'posee_cud']:
        # Unify 'Opción 1' and Nulls in 'desconocido'
        df[col] = df[col].replace({
            'nan': 'desconocido',
            'nulo (sin dato)': 'desconocido',
            'opción 1': 'desconocido', 
            'sin datos': 'desconocido'
        })

    elif col in ['hijos_pea', 'convivencia_pea']:
        # Imput Nulls to 'no'
        df[col] = df[col].replace({'nan': 'no',
                                   'nulo (sin dato)': 'no',
                                   'sin datos': 'no'})


# Verification

print("\n--- Count of 'diagnostico' (Post-Cleaning) ---")
print(df['diagnostico'].value_counts(dropna=False).to_markdown())

print("\n--- Count of Variable 'tratamiento' (Post-Cleaning) ---")
print(df['tratamiento'].value_counts(dropna=False).to_markdown())

print("\n--- Count of 'posee_cud' (Post-Cleaning) ---")
print(df['posee_cud'].value_counts(dropna=False).to_markdown())

print("\n--- Count of 'hijos_pea' (Post-Cleaning) ---")
print(df['hijos_pea'].value_counts(dropna=False).to_markdown())

print("\n--- Count of 'convivencia_pea' (Post-Cleaning) ---")
print(df['convivencia_pea'].value_counts(dropna=False).to_markdown())




--- Conteo de la Variable 'diagnostico' (Post-Limpieza) ---
| diagnostico        |   count |
|:-------------------|--------:|
| desconocido        |     127 |
| no                 |      66 |
| si                 |      39 |
| consum antidep.    |       2 |
| enfermedad crónica |       1 |

--- Conteo de la Variable 'tratamiento' (Post-Limpieza) ---
| tratamiento   |   count |
|:--------------|--------:|
| desconocido   |     141 |
| no            |      61 |
| si            |      33 |

--- Conteo de la Variable 'posee_cud' (Post-Limpieza) ---
| posee_cud   |   count |
|:------------|--------:|
| desconocido |     121 |
| no          |      98 |
| si          |      16 |

--- Conteo de la Variable 'hijos_pea' (Post-Limpieza) ---
| hijos_pea   |   count |
|:------------|--------:|
| no          |     131 |
| si          |     104 |

--- Conteo de la Variable 'convivencia_pea' (Post-Limpieza) ---
| convivencia_pea   |   count |
|:------------------|--------:|
| no                |     

### Analysis of Violence Descriptors

Cleaning of 'cant_personas_a_cargo' and character-to-number unification

In [ ]:
col_count = 'cant_personas_a_cargo'

# 1. Replace 'NO' for 0
df[col_count] = df[col_count].replace('NO', '0')

# 2. Replace Nulls for 0
df[col_count] = pd.to_numeric(df[col_count], errors='coerce') # Convierte cualquier otro valor anómalo que haya quedado a NaN
df[col_count] = df[col_count].fillna(0).astype(int)

# Variable Type Casting to Integer (int)
df['cant_personas_a_cargo'] = df['cant_personas_a_cargo'].astype(int)

# Verification
print("--- Count of 'cant_personas_a_cargo' ---")

print(df['cant_personas_a_cargo'].value_counts(dropna=False).to_markdown())

print(f"\nTotal Nulls in'cant_personas_a_cargo': {df['cant_personas_a_cargo'].isnull().sum()} nulos")

print("\n--- Type of variable 'cant_personas_a_cargo' (Post-Cleaning) ---")
df['cant_personas_a_cargo'].info()

--- Conteo de la Variable Limpia 'cant_personas_a_cargo' ---
|   cant_personas_a_cargo |   count |
|------------------------:|--------:|
|                       0 |      93 |
|                       2 |      47 |
|                       1 |      42 |
|                       3 |      34 |
|                       4 |      15 |
|                       6 |       3 |
|                       5 |       1 |

Total de Nulos en 'cant_personas_a_cargo': 0 nulos

--- Tipo de Variable 'cant_personas_a_cargo' (Post-Limpieza) ---
<class 'pandas.core.series.Series'>
RangeIndex: 235 entries, 0 to 234
Series name: cant_personas_a_cargo
Non-Null Count  Dtype
--------------  -----
235 non-null    int64
dtypes: int64(1)
memory usage: 2.0 KB


In [ ]:
# Cleaning of 'modalidad_de_violencia'

col_modalidad = 'modalidad_de_violencia'

# 2.1. Normalization
df[col_modalidad] = df[col_modalidad].astype(str).str.lower().str.strip()

# 2.2. Unification of Nulls
df[col_modalidad] = df[col_modalidad].replace({
    'nan': 'desconocida',
    'nulo (sin dato)': 'desconocida',
    'sin datos': 'desconocida'
})

# Verification
print("--- Count of 'modalidad_de_violencia' (Post-Cleaning)---")
print(df[col_modalidad].value_counts(dropna=False).to_markdown())

--- Conteo de la Variable 'modalidad_de_violencia' (Post-Limpieza)---
| modalidad_de_violencia   |   count |
|:-------------------------|--------:|
| doméstica                |     188 |
| desconocida              |      47 |


In [ ]:
# Cleaning of the 8 Types of Violence

violence_flags = [
    'fisica', 'psicologica', 'sexual', 'economica',
    'simbolica', 'ambiental', 'politica', 'digital'
]

for col in violence_flags:
    # Numeric Conversion of Columns (Handling 'True'/'False' -  1.0/0.0)
    # Coerce converts text such as 'Nulo (Sin dato)' or 'sin datos' to NaN
    df[col] = pd.to_numeric(df[col], errors='coerce')

    # Imputation of Nulls (NaN) to 0.0 
    df[col] = df[col].fillna(0.0)

    # Variable Type Casting to Integer (int)
    df[col] = df[col].astype(int)

# Verification
# I.E.
print("\n--- Verification of Nulls in 'fisica' (Post-Cleaning) ---")
print(f"Total of Nulls in 'fisica': {df['fisica'].isnull().sum()} nulos")
print(df['fisica'].value_counts().to_markdown())

print("\n--- Type of variable 'fisica' (Post-Cleaning) ---")
df['fisica'].info()


--- Verificación de nulos en 'fisica' (Post-Limpieza) ---
Total de Nulos en 'fisica': 0 nulos
|   fisica |   count |
|---------:|--------:|
|        0 |     119 |
|        1 |     116 |

--- Tipo de Variable 'fisica' (Post-Limpieza) ---
<class 'pandas.core.series.Series'>
RangeIndex: 235 entries, 0 to 234
Series name: fisica
Non-Null Count  Dtype
--------------  -----
235 non-null    int64
dtypes: int64(1)
memory usage: 2.0 KB


### Analysis of social variables

In [ ]:
# Cleaning of Social Variables

cols_sociales = ['personas_a_cargo', 'red_vincular']

for col in cols_sociales:

    # 1. Normalization
    df[col] = df[col].astype(str).str.strip().str.lower()

    # 2. Unification of Nulls
    df[col] = df[col].replace({
        'nan': 'desconocido',
        'nulo (sin dato)': 'desconocido',
        'sin datos': 'desconocido'
    })

    # 3. Correction of inconsistencies
   
    df[col] = df[col].str.replace('/', '_', regex=False)
    df[col] = df[col].str.replace(' ', '_', regex=False)


# Verification
print("--- Count of 'personas_a_cargo' (Post-Cleaning) ---")
print(df['personas_a_cargo'].value_counts(dropna=False).to_markdown())

print("\n--- Count of 'red_vincular' (Post-Cleaning) ---")
print(df['red_vincular'].value_counts(dropna=False).to_markdown())

--- Conteo de la Variable 'personas_a_cargo' (Post-Limpieza) ---
| personas_a_cargo   |   count |
|:-------------------|--------:|
| hija_hijo          |     151 |
| desconocido        |      52 |
| no                 |      29 |
| otros_as           |       3 |

--- Conteo de la Variable 'red_vincular' (Post-Limpieza) ---
| red_vincular              |   count |
|:--------------------------|--------:|
| desconocido               |      62 |
| amigas_os                 |      61 |
| parientes_no_convivientes |      55 |
| parientes_convivientes    |      45 |
| no                        |       9 |
| vecinas_os                |       3 |


### Analysis of the target variable 

In [ ]:
col_medida = 'medidas_de_proteccion'

# Normalization
df[col_medida] = df[col_medida].astype(str).str.lower().str.strip()

# Unification of Nulls
df[col_medida] = df[col_medida].replace({
    'nan': 'desconocida',
    'nulo (sin dato)': 'desconocida',
})

# Correction of inconsistencies
df[col_medida] = df[col_medida].str.replace('/', '_', regex=False)
df[col_medida] = df[col_medida].str.replace(' ', '_', regex=False)

# Verification
print("--- Verification of the cleaning (phase I) ---")
print("\n Count of 'medidas_de_proteccion':")
print(df[col_medida].value_counts().to_markdown())

--- Verificación de la Limpieza (FASE I) ---

Conteo de 'medidas_de_proteccion' (Limpieza de Texto/Nulos):
| medidas_de_proteccion                            |   count |
|:-------------------------------------------------|--------:|
| desconocida                                      |      93 |
| prohibición_de_acercamiento                      |      89 |
| no                                               |      40 |
| botón_antipánico__prohibición_acercamiento       |       6 |
| botón_antipánico                                 |       2 |
| prohibición_de_acercamiento__cese_de_hostigación |       2 |
| exclusión_del_hogar                              |       2 |
| cese_hostigamiento                               |       1 |


In [ ]:

col_fecha = 'fecha_fin_de_vigencia'

# 1. Normalization
# Convert to string to look for paterns
df[col_fecha] = df[col_fecha].astype(str).str.lower().str.strip()
df_non_null = df[df[col_fecha] != 'nan'].copy()

# 2. Identifying values that are NOT date codes (Non-numeric values)
# When converting the column to numeric, errors (coerce) represent the string values
non_numeric_values = pd.to_numeric(df_non_null[col_fecha], errors='coerce')
non_numeric_strings = df_non_null[non_numeric_values.isna()][col_fecha]

# 3. Count the frequency of each unique string
print("--- Unique strings in 'fecha_fin_de_vigencia' that are not numeric codes ---")
if not non_numeric_strings.empty:
    print(non_numeric_strings.value_counts().to_markdown())
else:
    print("No invalid date strings were found beyond those already processed.")

--- Strings Únicos en 'fecha_fin_de_vigencia' que NO son Códigos Numéricos ---
| fecha_fin_de_vigencia   |   count |
|:------------------------|--------:|
| hasta junio             |       1 |


In [ ]:
REPLACEMENT_CODE = 45809 # Excel Code for 01/06/2025

# 1. Normalization
df[col_fecha] = df[col_fecha].astype(str).str.lower().str.strip()

# 2. Apply replacement using pattern matching (substring search)
# Using .str.contains('hasta') & .str.contains('junio') to ensure robustness against internal spacing (e.g., 'hasta junio')
df.loc[df[col_fecha].str.contains('hasta', na=False) & df[col_fecha].str.contains('junio', na=False), col_fecha] = str(REPLACEMENT_CODE)

# 3. Convert the column to numeric (completes the cleaning process)
df[col_fecha] = pd.to_numeric(df[col_fecha], errors='coerce')

# Verification:
count_after = (df[col_fecha] == REPLACEMENT_CODE).sum()

print(f"Cases converted to a 45809: {count_after}")

In [ ]:
FECHA_LIMITE_INFERIOR = 45474 #Excel Code for 01/07/2024

# Values below 45474 (outdated/expired dates) are forced to NaN
df.loc[df[col_fecha] < FECHA_LIMITE_INFERIOR, col_fecha] = np.nan

# Verification
print("--- Count of 'fecha_fin_de_vigencia' ---")
print(f"Total valid numeric values:: {df[col_fecha].notna().sum()}")
print(df[col_fecha].describe().to_markdown())

--- Conteo Final de 'fecha_fin_de_vigencia' (Post-Filtro de Rango) ---
Total de valores numéricos válidos: 63
|       |   fecha_fin_de_vigencia |
|:------|------------------------:|
| count |                  63     |
| mean  |               45896     |
| std   |                 197.344 |
| min   |               45673     |
| 25%   |               45790.5   |
| 50%   |               45867     |
| 75%   |               45923.5   |
| max   |               46984     |


In [ ]:
# ----------------------------------------------------------------------
# CLEAN AND IMPUTE the column 'denuncio'
# ----------------------------------------------------------------------

col_denuncio = 'denuncio'

# A. Standarize text
df[col_denuncio] = df[col_denuncio].astype(str).str.lower()

# B. Replace strings representing empty or null values with actual NaN
df[col_denuncio] = df[col_denuncio].replace(['sin datos', 'sin dato', ' ', '', 'nan'], np.nan)


# Logical Flags for Business Rules
## FECHA_LIMITE_INFERIOR = 45474 # Excel code for 07/01/2024

#Flag A: Valid Protection Measure(not 'no' or 'desconocida')
medida_valida_logica = ~df[col_medida].isin(['no', 'desconocida'])

# Bandera B: The date is valid (not 'Nulo' and within the rank >= 01/07/2024)
fecha_valida_logica = df[col_fecha].notna() & (df[col_fecha] >= FECHA_LIMITE_INFERIOR)


# Rules fot the target 

# a: If 'denuncio' is NULL AND (Valid Measure) AND (Valid Date), impute to 'YES'.
df.loc[
    (df[col_denuncio].isna()) &
    (medida_valida_logica == True) &
    (fecha_valida_logica == True),
    col_denuncio
] = 'si'

# b: Impute the remaining Nulls to 'NO' 
df[col_denuncio] = df[col_denuncio].fillna('no')


# verification

print("--- Values after cleaning and imputing ( ---")
print(df[col_denuncio].value_counts().to_markdown())

--- Valores después de limpiar e imputar (Post-Regla) ---
| denuncio   |   count |
|:-----------|--------:|
| si         |     140 |
| no         |      95 |


#### Results

Closing of Data Wrangling: the cleaning results are verified. All variables have the correct data types assigned, and null values have been handled consistently with the scope of the data cleaning stage. The two outliers will be addressed during Feature Engineering.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235 entries, 0 to 234
Data columns (total 36 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   profesionales                      235 non-null    object 
 1   edad                               225 non-null    float64
 2   motivo_de_consulta                 235 non-null    object 
 3   medio_por_el_que_ingresa           235 non-null    object 
 4   genero                             235 non-null    object 
 5   nacionalidad                       235 non-null    object 
 6   barrio                             235 non-null    object 
 7   municipio                          235 non-null    object 
 8   localidad                          235 non-null    object 
 9   estado_civil                       235 non-null    object 
 10  nivel_educativo                    235 non-null    object 
 11  situacion_laboral                  235 non-null    object 

# Export cleaned dataset to CSV

In [ ]:
# 1. Save
df.to_csv('dataset_limpio_ok.csv', index=False)

print("Archivo 'dataset_limpio_ok.csv' generado con éxito en tu entorno de Colab.")

# 2.Download
from google.colab import files
files.download('dataset_limpio_ok.csv')

print("Descarga iniciada a tu computadora.")

Archivo 'dataset_limpio_ok.csv' generado con éxito en tu entorno de Colab.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descarga iniciada a tu computadora.
